# **Week 7 Assignment: Retrieval-Augmented Generation (RAG) System**

Rudra Mehta

# Step 1: Import Libraries

In [86]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu pypdf transformers sentence-transformers accelerate

In [87]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import pipeline

# Step 2. Document Ingestion

The first step in the RAG pipeline is document ingestion. The system accepts different types of documents, such as PDF, TXT, HTML, and other supported file formats. These documents are loaded using the appropriate document loader (for example, PyPDFLoader for PDF files and TextLoader for text files). The extracted text is then used as the knowledge base to answer the user's questions.

In [88]:
from google.colab import files

uploaded = files.upload()

Saving What is Retrieval.pdf to What is Retrieval (2).pdf


In [89]:
file_path = "/content/What is Retrieval.pdf"

if file_path.endswith(".pdf"):
    loader = PyPDFLoader(file_path)

elif file_path.endswith(".txt"):
    loader = TextLoader(file_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 5


# Step 3: Text Chunking
The extracted text is divided into smaller pieces called chunks using a text splitter such as RecursiveCharacterTextSplitter. Breaking the text into smaller chunks helps the system find the most relevant information more accurately and improves the quality of the answers.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 20


# Step 4: Text Embedding

Each text chunk is converted into a numerical vector called an embedding using a pre-trained embedding model. These embeddings capture the meaning of the text, making it possible to compare the user's question with the stored document chunks. In this project, the sentence-transformers/all-MiniLM-L6-v2 model is used to generate embeddings for efficient and accurate similarity search.

In [91]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Step 5: Vector Database
The generated embeddings are stored in a vector database, which helps the system quickly find the most relevant document chunks. In this project, FAISS is used as the vector database. Other popular vector databases include Chroma, Milvus, and Pinecone. When a user asks a question, the vector database compares the query embedding with the stored embeddings and retrieves the most similar document chunks.

In [ ]:
vector_db = FAISS.from_documents(chunks,embeddings)

print("Vector Database Created Successfully")

Vector Database Created Successfully


# Step 6: Load LLM
Objective

A pre-trained Large Language Model (LLM) is loaded to generate answers. In this project, the TinyLlama-1.1B-Chat model from Hugging Face is used. The model takes the retrieved context and the user's question as input, then generates a clear and relevant answer based on the provided information.

In [107]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto"
)

print("LLM Loaded Successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM Loaded Successfully


# Step 7: User Query Processing

When a user submits a question, the system converts the query into an embedding using the same embedding model that was used for the document chunks. Using the same embedding space ensures meaningful similarity comparisons between the query and stored document vectors.

In [118]:
query = input("Ask your question: ")

Ask your question: working


# Step 8: Context Retrieval

The vector database performs a similarity search between the query embedding and stored document embeddings. 
Based on similarity scores, the system retrieves the top-k most relevant text chunks. 
These retrival chunks provide information to generate accurate answers.

In [119]:
docs = vector_db.similarity_search(
    query,
    k=8
)

print("\nRetrieved Documents\n")

for i, doc in enumerate(docs, start=1):
    print("="*80)
    print(f"Chunk {i}")
    print("="*80)
    print(doc.page_content)
    print()


Retrieved Documents

Chunk 1
time or scheduled so the system always retrieves latest information. 
What Problems does RAG solve 
1. Hallucinations: Traditional generative models can produce incorrect information. 
RAG reduces this risk by retrieving verified, external data to ground responses in 
factual knowledge.

Chunk 2
the model’s effectiveness. 
4. Bias and Fairness: It can inherit biases present in the training data or retrieved 
documents, necessitating ongoing efforts to ensure fairness and mitigate biases. 
RAG Applications 
1. Question-Answering Systems: It enables chatbots or virtual assistants to pull 
information from a knowledge base or documents and generate accurate, context 
aware answers.

Chunk 3
4. Information Retrieval: Goes beyond traditional search by retrieving documents and 
generating meaningful summaries of their content.

Chunk 4
costs and computational load. 
6. Scalability Across Domains: It is adaptable to diverse industries from healthcare to 
finance 

# Step 9: Create Context

The retrieved document chunks are combined into a single block of text. Each chunk is separated by a blank line to make it easier to read. This combined context is then provided to the language model, which uses it as the main source of information to generate an accurate answer to the user's question.

In [120]:
context = "\n\n".join(
    [doc.page_content for doc in docs]
)

# Step 10: Create Prompt
Objective

A prompt is created by combining the retrieved document content with the user's question. The prompt tells the language model to answer only using the provided information. If the answer is not available in the retrieved context, the model responds that the information was not found instead of making up an answer.

In [121]:
prompt = f"""
You are a helpful assistant.

Use ONLY the information provided in the context.

If the answer is not present in the context, reply exactly:

Answer not found in the document.

Context:
{context}

Question:
{query}

Give a complete, detailed answer.
"""

# Step 11: Answer Generation
The system combines the user's question with the most relevant information found in the documents. This combined input is then sent to the language model, which generates an answer based on the retrieved information. Since the answer comes from the documents, it is more accurate and reduces the chances of incorrect or made-up information.

In [122]:
response = generator(
    prompt,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.1,
    top_p=0.95,
    eos_token_id=generator.tokenizer.eos_token_id,
    return_full_text=False
)

print("\nGenerated Answer:\n")
print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Answer:


Answer:
RAG is a powerful tool for generating accurate and context-aware responses. It combines 
retrieval and generation to use external data for more factual and context-aware responses. 
RAG can be used in various applications, including question-answering systems, information 
retrieval, and chatbots. The model's effectiveness depends on the quality of the retrieved 
documents, which can be improved by augmenting the LLM prompt. The model's complexity is 
addressed by careful tuning and optimization, and the latency is mitigated by combining 
retrieval and generation. The model's accuracy is enhanced by incorporating external data 
and embeddings, which are refreshed regularly. The model's output is context-aware and 
grounded in reliable data, making interactions more informative and personalized.


# Conclusion

The Retrieval-Augmented Generation (RAG) system helps answer questions by using information from your own documents instead of relying only on the AI's knowledge. It works by reading documents, splitting them into smaller parts, converting them into embeddings, storing them in a vector database, finding the most relevant information, and then generating accurate answers using a language model. This makes the responses more reliable and useful for tasks like document-based question answering, research, enterprise knowledge management, intelligent search, and AI chatbots.